# <u>Homework:</u> Solving the Knapsack Problem


Remark: Commenting your code will be part of the evaluation.



**Names** of each member of the group (3 members max), together with percentages of participation: 

(*example*: Richard Bellman (30%) Giselle Beyoncé (68%) Roland Grappe (2%))



## Description of the Context: Bounded Knapsack Problem

You are given a set of items, each with an associated value and weight, and a knapsack with a maximum weight capacity. Each item is available a certain number of times. The goal is to determine the most valuable combination of items to place in the knapsack without exceeding its weight capacity.

**Problem Parameters:**

- **Set of Items:** There is a finite set of items, often denoted as $I = \{1, 2, \ldots, n\}$.

- **Item Properties:**
  - Each item $i$ has an associated value $v_i$ (positive integer number), representing the profit gained by including the item in the knapsack.
  - Each item $i$ has an associated weight $w_i$ (positive integer number), indicating the weight of the item.
  - Each item $i$ has an associated maximum quantity $b_i$ (a non-negative integer), specifying the maximum number of times that item can be selected.

- **Knapsack Capacity:** The knapsack has a maximum weight capacity, denoted as $W$ (a positive integer number).

**Objective:**

The objective of the bounded knapsack problem is to determine a selection of items, taking into account the quantity bounds, that maximizes the total value of the selected items while ensuring that the total weight does not exceed the knapsack's capacity.

A solution should be represented by a list containing the selected quantity of each item, in the same order as the input lists. In the fractional version, these quantities may be nonintegral.


## I. Fractional Bounded Knapsack

In the fractional version of the problem, you are allowed to take arbitrary fractions of items.

**Q1.** Write a function `knapfrac(values, weights, quantities, capacity)` whose arguments are:
- a list of item values,
- a list of item weights,
- a list of upper bounds on the quantities of each item,
- an integer capacity (the maximum weight the knapsack can hold),

and which returns the maximum value of a fractional knapsack.

In [ ]:
def knapfrac(values, weights, quantities, capacity):
    n = len(values)

    # We first compute the ratios, in order to determine which item is the most profitable per unit of weight
    ratios = []
    for i in range(n):
        ratios.append(values[i] / weights[i])

    # We order the items in decreasing order of ratios
    items_sorted_by_ratio_index = []
    ratios_temp = ratios.copy()  # To not modify the initial ratios information
    indices_temp = list(range(n))  # To preserve the original indices
    nb_different_values = len(set(ratios))  # To avoid useless iterations
    for i in range(nb_different_values):
        m = max(ratios_temp)
        for j in range(len(ratios_temp) - 1, -1, -1):
            if ratios_temp[j] == m:
                items_sorted_by_ratio_index.append(indices_temp[j])
                del ratios_temp[j]
                del indices_temp[j]

    # Now we take items in the order computed previously
    total_sum = 0
    copy_quantities = quantities.copy() # To not modify the initial quantities informations
    while (capacity > 0 and len(items_sorted_by_ratio_index) != 0):
        # We check if it remains quantities of the first item
        temp_index = items_sorted_by_ratio_index[0]
        if (copy_quantities[temp_index] == 0):
            del items_sorted_by_ratio_index[0]
        index_of_added = items_sorted_by_ratio_index[0]
        # We now verify if the weight of the item to be added does not exceed the total capacity
        if (weights[index_of_added] <= capacity):
            capacity -= weights[index_of_added]
            total_sum += values[index_of_added]
            copy_quantities[index_of_added] -= 1
        # If the remaining capacity is not sufficient to put the entire item
        else: 
            quantity_added = capacity / weights[index_of_added]
            total_sum += values[index_of_added] * quantity_added

    return total_sum

**Q2.** Enhance the previous function so that it also outputs an associated solution of optimum value.

In [ ]:
#Write your code here

**Q3.** Test your implementation with suitable choices of values, weights, quantities, and capacities to ensure that it works correctly.

In [ ]:
#Write your code here

## II. Bounded Knapsack by Dynamic Programming

**Q1.** Provide and explain a recurrence relation satisfied by the bounded knapsack problem.

**Answer:** 

*[ thoughts, i will leave a trace of my reasoning steps and what i have tried]:*
*I need to be sure of the defenition of the bounded problem, here i assume it just like the fractional one but the decision variables (number of items i choose per item) is a natural number instead of a real number. my decision variables will be called X_i (i because one per item).*

We are then working on maximixing the following function: **SUM(i=1 to n)_{ X_i * v_i }**
While these constraints are true:

- for all i :  **0 <= X_i <= b-i** 
- **SUM(i=1 to n)_{ X_i * w_i } <= W**
- for all i : X_i is in the natural number set

This exercice asks us to work in Dynamic Programming so we need to find a recursive relation in this problem. My first instict is to recurse on the number of items, it seems like the only parameter we can recurse with maybe the other being Weight. I will choose to explore by the number of items (fixing W to its base value).

- After working on the problem my it seems more easy to recurse on the Weights. because if i know the best combinations for W=1 to 4 then for W=5 i have to only check the new bigger items and the combinations of 1+4 and 2+3 to see wich is better. 

the above approach does not seem to work because of the bounded values, in theory it works but we need to keep a lot of information between the recursive steps, i will search for something else.

Well the idea may be as simple as to recurse on 2 parameters, the number of items and i, i recurse on the weight used by each item, that would be the same as to track X_i or to recurse on W too. 
In class we saw the Rod problem, we can use a similar analogy, i suppose that i alredy know the best value cases for my previous "smaller cuts" of the rod, then i check all the possibles values X_i (named k_j) for the new i-th item and for each fill the reminder r_j of the rod r_j = (W - X_i*k_j) with the best case that i already know for that "lenght" thus completing the recursive relation without having to pass the bi information backwards since it is solved independently at each step.

The complexity is very direct to calculate, we have to check the max between the at worst bi + 1 values for each then we have to that for each Weight. So the complexity comes at

**O(W * SUM(i=1 to n){ bi + 1})**

---------

I asked the Ai gemini about it thoughts about my solution (you can find in annex at the end of the subject the prompt) it proposed 2 things: a way to reduce complexity by calculating the solutions for the powers of 2 and using the base 2 to reduce complexity to 

**O(W * SUM(i=1 to n){ log(bi) } )**

it also proposed a new solution it named: Sliding Window DP, that is even better using the fact that we also care about the reminders (modulo) of the w_i, since we only transition from one state to another if the 'hole' left by the cut is exactly a size of a multiple of the weight. managing to reduce the complexity to 

**O(W*n)**

-----------

In the end i will be implemting my own solution and a second version using the binary optimization to test how much faster it gets, i will not implement the optimal solution since i would not have found it easily by myself. 



**Q2.** Write a recursive function `knapdyn(values, weights, quantities, capacity)` implementing the previous recurrence, without memoization, and returning the maximum value of a bounded knapsack problem.

In [ ]:
#it says not to use memoization, at first i implemented a bottom-up approach without thinking, but that is not a recursive function....
#i have left it here to later test the speeds, renamed to knapdyn_bottom_up.

#i will code the items starting at 0 because arrays.

def knapdyn_bottom_up(values, weights, quantities, capacity):
    n = len(values)
    #we need a 2D array for items and capacity, 
    # with Array[i][w] = max value for w and only using items 0 to i-1 
    Array = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(n):
        for w in range(capacity + 1):
            #Max items i we can fit into w
            if weights[i] <= w:
                #i assume c_w will never be 0 since we said w_i are positive integer numbers at the start.
                max_count = min(quantities[i], w // weights[i])
            else:
                max_count = 0

            # Find the best value
            max_value = 0
            for q in range(max_count + 1):
                total_value = Array[i][w - q * weights[i]] + q*values[i]
                if total_value > max_value:
                    max_value = total_value
            Array[i + 1][w] = max_value

    return Array[n][capacity]



#the actual function that was asked for: 
#base case are 0, and looking at the bottom up implementation i can catch the case where
#the item cannot fit at all, and skip it. So it was not a full waste of time 


#i had a problem, i don't want to use array[:-1] since it creates new arrays. so i used 'i' as an argument but
#then the user could touch the value i while calling the function and breaking it so i made a second hidden function inside.


def knapdyn(values, weights, quantities, capacity):
    #since defined inside the values, weights and quantites are known.
    def true_recurse(capacity, i):
        
        # Base case: no items left or no capacity
        if i < 0 or capacity == 0:
            return 0

        #Current item, we are working on the last item on the list generally
        c_val = values[i]
        c_w = weights[i]
        c_q = quantities[i]

        #If the item does not fit, skip! 
        if c_w > capacity:
            return true_recurse(capacity, i - 1)

        #if it fits we do the same as the bottom up: we loop on the possible quantities 
        max_value = 0
        #i assume c_w will never be 0 since we said w_i are positive integer numbers at the start.
        max_count = min(c_q, capacity // c_w)

        for q in range(max_count + 1):  
            next_capacity = capacity - q * c_w

            #recurse call for the remaining capacity= W - q*c_w and we use the remaining items (cutting the last item = current item)
            recurse_bestvalue = true_recurse(next_capacity, i - 1)
            total_value = q * c_val + recurse_bestvalue

            max_value = max(max_value, total_value)

        return max_value

    #i call the "protected function"
    return true_recurse(capacity,len(values)-1)

**Q3.** Enhance `knapdyn` into an efficient function `knapdynMem` which uses memoization.

In [ ]:
#now that we have the top-bottom implementation it is easy to add the memoization with just a dictionary.
#i littaraly copy pasted and added the check dico[key] and next to all the returns i added the dico[key] = result.

#the fact that i have a hidden function, makes it easy to add an internal dictionary that is not global.


def knapdynMem(values, weights, quantities, capacity):
    dico = {}
    #since defined inside the values, weights and quantites are known.
    def true_recurse(capacity, i):

        # Create a key for the current item and capacity
        key = (i, capacity)
    
        # Check if the result is alredy in dico 
        if key in dico:
            return dico[key]
        
        # Base case: no items left or no capacity
        if i < 0 or capacity == 0:
            dico[key] = 0
            return 0

        #Current item, we are working on the last item on the list generally
        c_val = values[i]
        c_w = weights[i]
        c_q = quantities[i]

        #If the item does not fit, skip! 
        if c_w > capacity:
            result = true_recurse(capacity, i - 1)
            dico[key] = result
            return result

        #if it fits we do the same as the bottom up: we loop on the possible quantities 
        max_value = 0
        #i assume c_w will never be 0 since we said w_i are positive integer numbers at the start.
        max_count = min(c_q, capacity // c_w)

        for q in range(max_count + 1):  
            next_capacity = capacity - q * c_w

            #recurse call for the remaining capacity= W - q*c_w and we use the remaining items (cutting the last item = current item)
            recurse_bestvalue = true_recurse(next_capacity, i - 1)
            total_value = q * c_val + recurse_bestvalue

            max_value = max(max_value, total_value)

        result = max_value
        dico[key] = result
        return result

    #i call the "protected function"
    return true_recurse(capacity,len(values)-1)

**Q4.** Modify the previous functions so that they also output an associated solution of optimum value.

In [ ]:
#there is a simple way but it uses the .copy for the arrays since we don't want to accidently modify the optimum solution
#but making copies adds complexity, in this case i will use it since we will be testing the other function and not this one.
#thus keeping the comparison fair in terms of actual complexity and not added things, the same reason why i avoided the [:-1].


#copy paste + small changes return is now a tuple 


def knapdynMem_solprint(values, weights, quantities, capacity):
    dico = {}
    n = len(values)
    #since defined inside the values, weightq and quantites are known.
    def true_recurse(capacity, i):

        # Create a key for the current item and capacity
        key = (i, capacity)
    
        # Check if the result is alredy in dico 
        if key in dico:
            return dico[key]
        
        # Base case: no items left or no capacity
        if i < 0 or capacity == 0:
            result = (0, [0 for _ in range(n)])
            dico[key] = result
            return result

        #Current item, we are working on the last item on the list generally
        c_val = values[i]
        c_w = weights[i]
        c_q = quantities[i]

        #If the item does not fit, skip! 
        if c_w > capacity:
            result = true_recurse(capacity, i - 1)
            dico[key] = result
            return result

        #if it fits we do the same as the bottom up: we loop on the possible quantities 
        max_value = 0
        #i assume c_w will never be 0 since we said w_i are positive integer numbers at the start.
        max_count = min(c_q, capacity // c_w)
        solution = [0 for _ in range(n)]

        for q in range(max_count + 1):  
            next_capacity = capacity - q * c_w

            #recurse call for the remaining capacity= W - q*c_w and we use the remaining items (cutting the last item = current item)
            recurse_bestvalue , recurse_solution = true_recurse(next_capacity, i - 1)
            total_value = q * c_val + recurse_bestvalue

            max_value = max(max_value, total_value)
            
            #oups, only update the solution if we found a new max_value
            if total_value == max_value:
                solution = recurse_solution.copy()
                solution[i] = q

        result = (max_value, solution)
        dico[key] = result
        return result

    #i call the "protected function"
    return true_recurse(capacity,len(values)-1)



**Q5.** Devise a suitable set of test instances, and use them to compare the performances of your functions with `timeit`.

In [ ]:


import time 

## III. Knapsack by Branch and Bound

Branch and bound is a general algorithmic technique used for solving optimization problems, including the knapsack problem. It systematically explores a search tree of potential solutions, pruning branches that cannot lead to an optimal solution.

The primary steps in the branch and bound approach are as follows:

- **Branching:** The problem space is divided into smaller subproblems. For example, in the bounded knapsack problem, branching can consist of choosing how many copies of a given item to include, from zero up to its maximum allowed quantity.

- **Bounding:** At each node in the search tree, an upper bound on the optimal solution is computed. If the bound is no greater than the value of the best feasible solution found so far, the branch can be pruned.

- **Backtracking:** The algorithm explores promising branches while keeping track of the best solution found so far. When it reaches a leaf node, it updates the best solution if a better one is found.

**Q1.** Which of the previous functions can be used to obtain an upper bound at each node?

**Answer:**


**Q2.** Write a function `knapBnB(values, weights, quantities, capacity)` which returns the maximum value of a bounded knapsack problem using a branch-and-bound approach.

In [ ]:
#Write your code here

**Q3.** Test your implementation with suitable choices of values, weights, quantities, and capacities to ensure that it works correctly. Whenever possible, compare the results of `knapBnB` with those of `knapdynMem` to check their correctness.

In [ ]:
#Write your code here

**Q4.** Using `timeit`, compare the performances of `knapBnB` and `knapdynMem`.

In [ ]:
#Write your code here

## IV. Complexity Analysis

With explanations, give the respective complexities of `knapfrac`, `knapdyn`, `knapdynMem`, and `knapBnB`.

Are the previous practical comparisons consistent with these theoretical results? Explain which characteristics of the instances influence the performances of the different algorithms.

**Answer:** (Write your text here, or incorporate a photo of your written answer)



## V. Unbounded Knapsack

In the unbounded version, there are no limitations on the quantity of each item that can be taken.

**Q1.** How can the previous functions be used to solve the unbounded knapsack version?

**Answer:** (Write your text here, or incorporate a photo of your written answer)


## VI. Use of Conversational Agents (ChatGPT, etc.)

If you used conversational agents while completing this homework, explain for which tasks you used them and how you verified the correctness of their suggestions.

- thoughts on my recusive approach.
- grammar correction of my english
- find code mistakes.

